# UWCT Model A final test (D-043): E027 on the v1 test and on Zenodo `sk`

**What this notebook does:** the one final test evaluation of **E027** (yolo26l, round-2 combination C4, dataset v4),
declared in `experiments/final_candidates.csv` (D-043) before this notebook ran. Two separate targets, each evaluated
once by `scripts/final_test_eval.py` (CLAUDE.md rule 7):

1. `--target test`: the 348 v1 test frames (`data.yaml`): new sessions at known sites.
2. `--target test_zenodo_sk`: the 428 held-out Zenodo `sk` frames (`data_test_zenodo_sk.yaml`): an external new site.

For both, the operating confidence is chosen on **val** first (max F1), then frozen. Results never feed back into
training, thresholds or model choice. No training happens here.

**Inputs to attach (Add Input):** the 4 archive datasets used by rounds 1-2, and the **output of notebook
`uwct-tuning-e011-e019`** (version "D-040 round 2 E020-E029 (97810a5)"), which holds E027's run folder and best.pt.

**Settings:** GPU T4 (one GPU is used), Internet on, secret `GITHUB_TOKEN`, environment pinned as in round 2.
Save & Run All (Commit). Expected total: about 25-40 min.

Cells 1-10 are the round-2 setup cells (archives verified, raw check, pinned install, v4 view built and VERIFIED);
only Cell 2 changed (it skips non-folder entries, since a notebook output is attached too).

In [ ]:

# Cell 1 - inspect the environment (read-only)
# 1) Where Kaggle mounted our 4 datasets, and the files inside each
# 2) Which GPU we got and how much memory it has
# 3) How much free disk space /kaggle/working and /tmp have (we must unzip ~17 GB)

import os, subprocess, time

# notebook start time: Cell 12's deadline (D-038 point 2) is measured from here
os.makedirs('/kaggle/working/uwct_logs', exist_ok=True)
with open('/kaggle/working/uwct_logs/t0.txt', 'w') as fh:
    fh.write(str(time.time()))
print('run type:', os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'unknown'))

print("=== INPUTS ===")
print(subprocess.run("ls -la /kaggle/input/*", shell=True, capture_output=True, text=True).stdout)

print("=== GPU ===")
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)

print("=== DISK ===")
print(subprocess.run("df -h /kaggle/working /tmp", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Cell 2 - inspect the attached datasets (read-only)
# For each dataset mounted under /kaggle/input/<owner>/<dataset>/ we print
# at most 15 entries, so the output stays small even if a dataset has
# thousands of files. We also count the files in each one.

import os

INPUT_ROOT = "/kaggle/input"

for owner in sorted(os.listdir(INPUT_ROOT)):
    owner_dir = os.path.join(INPUT_ROOT, owner)
    if not os.path.isdir(owner_dir):
        print(f"=== {owner} (file)")
        continue
    for ds in sorted(os.listdir(owner_dir)):
        ds_dir = os.path.join(owner_dir, ds)
        if not os.path.isdir(ds_dir):
            print(f"=== {owner}/{ds} (file)")
            continue
        n_files = sum(len(files) for _, _, files in os.walk(ds_dir))
        print(f"=== {owner}/{ds}   ({n_files} files in total)")
        for i, name in enumerate(sorted(os.listdir(ds_dir))):
            if i >= 15:
                print("    ... (more entries not shown)")
                break
            path = os.path.join(ds_dir, name)
            if os.path.isfile(path):
                print(f"    {name}   {os.path.getsize(path):,} bytes")
            else:
                print(f"    {name}/")
        print()

In [ ]:
# Cell 3 - locate our 4 uploaded archives and check their sizes (read-only)
# Expected sizes are the byte counts of the verified local copies (2026-11-06).
# A size match is a quick first check; the sha256 check comes in the next cell.

import os

OUR_ROOT = "/kaggle/input/datasets/sakshampainuly007"
EXPECTED_BYTES = {
    "biology.zip.bin":                5_478_527_588,
    "harsh_model_a_2026-11.zip.bin": 10_121_406_479,
    "harsh_v1_extra_2026-11.zip.bin":  1_819_177_935,
    "harsh_v1_colour_2026-11.zip.bin":   263_128_178,
}

found = {}
for dirpath, _, files in os.walk(OUR_ROOT):
    for f in files:
        found[f] = os.path.join(dirpath, f)

for name, expected in EXPECTED_BYTES.items():
    if name not in found:
        print(f"MISSING   {name}")
        continue
    size = os.path.getsize(found[name])
    status = "OK      " if size == expected else "MISMATCH"
    print(f"{status}  {found[name]}")
    print(f"          size {size:,} B (expected {expected:,} B)")

extra = set(found) - set(EXPECTED_BYTES)
if extra:
    print("Unexpected files:", sorted(extra))

In [ ]:
# Cell 4 - verify archive integrity with sha256
# Expected hashes were computed on the verified local copies before upload
# (D:\uwct_kaggle_upload\*\*.sha256). All 4 must print MATCH before we unpack.
# The result is also saved to /kaggle/working/uwct_logs/archive_sha256.txt,
# which is kept when the notebook version is saved.

import hashlib, os, time

ARCHIVES = {
    "/kaggle/input/datasets/sakshampainuly007/uwct-biology-raw/biology.zip.bin":
        "88d3ba76e3e9ed55af7929667e8e3abbfb51f6d7444130c27627cbc3494899b3",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-model-a-2026-11/harsh_model_a_2026-11.zip.bin":
        "ecbcb2a58403365b79c6764d714d414ca97a0e4c80d088db43518f76b76dc2c3",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-v1-extra-2026-11/harsh_v1_extra_2026-11.zip.bin":
        "02ae7b10f23a6f5c42b8598039b3a74c7e0ded47d3bcea5d124f1b3e4e606cbd",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-v1-colour-2026-11/harsh_v1_colour_2026-11.zip.bin":
        "405c373d1d3680b98433ab537c83e73fd8a961a607ae616415d6cfaf2c1852e7",
}

LOG_DIR = "/kaggle/working/uwct_logs"
os.makedirs(LOG_DIR, exist_ok=True)

def sha256_of(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

lines, all_ok = [], True
for path, expected in ARCHIVES.items():
    t0 = time.time()
    got = sha256_of(path)
    ok = got == expected
    all_ok &= ok
    line = f"{'MATCH   ' if ok else 'MISMATCH'}  {os.path.basename(path)}  {got}  ({time.time() - t0:.0f} s)"
    print(line)
    lines.append(line)

summary = "ALL 4 ARCHIVES VERIFIED" if all_ok else "STOP: at least one archive does not match"
print("\n" + summary)
with open(os.path.join(LOG_DIR, "archive_sha256.txt"), "w") as f:
    f.write("\n".join(lines + ["", summary]) + "\n")

In [ ]:
# Cell 5 - unpack the verified archives to /tmp/uwct_raw (raw data, treated read-only)
# /tmp is used because /kaggle/working is only 20 GB; /tmp is wiped when the
# session ends, so this cell must be re-run in every new session.
# Each archive has one top folder named like our repo's data/raw/<folder>.
# Safety: every entry is checked BEFORE extraction (no absolute paths, no "..",
# correct top folder). Nothing is overwritten: an existing target stops the cell.

import os, zipfile, shutil, time

RAW_DIR = "/tmp/uwct_raw"
IN = "/kaggle/input/datasets/sakshampainuly007"
ARCHIVES = {   # archive path -> expected top folder
    f"{IN}/uwct-biology-raw/biology.zip.bin":                              "biology",
    f"{IN}/uwct-harsh-model-a-2026-11/harsh_model_a_2026-11.zip.bin":      "harsh_model_a_2026-11",
    f"{IN}/uwct-harsh-v1-extra-2026-11/harsh_v1_extra_2026-11.zip.bin":    "harsh_v1_extra_2026-11",
    f"{IN}/uwct-harsh-v1-colour-2026-11/harsh_v1_colour_2026-11.zip.bin":  "harsh_v1_colour_2026-11",
}

os.makedirs(RAW_DIR, exist_ok=True)
log = []
for archive, top in ARCHIVES.items():
    target = os.path.join(RAW_DIR, top)
    if os.path.exists(target):
        raise SystemExit(f"STOP: {target} already exists - not overwriting. "
                         f"Restart the session or remove it deliberately.")
    t0 = time.time()
    with zipfile.ZipFile(archive) as zf:
        members = zf.infolist()
        bad = [m.filename for m in members
               if m.filename.startswith("/") or ".." in m.filename.split("/")
               or m.filename.split("/")[0] != top]
        if bad:
            raise SystemExit(f"STOP: unsafe or unexpected entries in {archive}: {bad[:5]}")
        zf.extractall(RAW_DIR)
    n_files = sum(1 for m in members if not m.is_dir())
    line = f"{top:26s} {n_files:6d} files extracted   ({time.time() - t0:.0f} s)"
    print(line)
    log.append(line)

total, used, free = shutil.disk_usage(RAW_DIR)
disk = f"/tmp free after extraction: {free / 1e9:.0f} GB"
print("\n" + disk)
with open("/kaggle/working/uwct_logs/extraction.txt", "w") as f:
    f.write("\n".join(log + ["", disk]) + "\n")

In [ ]:
# Cell 6 - clone the project repository (private) at the integration branch p4/colab-baseline
# The token is read from Kaggle Secrets and passed as a one-off HTTP header,
# so it is NOT written into the repo's .git/config and never printed.
# The exact commit used is printed and logged, so every result in this
# notebook can be traced back to the code that produced it.

import base64, os, subprocess
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/SakshamPainuly/underwater-creature-tracking.git"
BRANCH = "p4/colab-baseline"
REPO_DIR = "/tmp/uwct_repo"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()

if os.path.exists(REPO_DIR):
    raise SystemExit(f"STOP: {REPO_DIR} already exists - not overwriting.")

r = subprocess.run(
    ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}",
     "clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO_DIR],
    capture_output=True, text=True)
del token, auth
if r.returncode != 0:
    # print git's message with any secret material removed
    raise SystemExit("STOP: clone failed:\n" + r.stderr.replace(REPO_URL, "<repo>"))

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args],
                          capture_output=True, text=True).stdout.strip()

info = [
    f"repo    : {REPO_URL}",
    f"branch  : {git('rev-parse', '--abbrev-ref', 'HEAD')}",
    f"commit  : {git('rev-parse', 'HEAD')}",
    f"message : {git('log', '-1', '--format=%s (%ad)', '--date=short')}",
    f"dirty   : {'yes' if git('status', '--porcelain') else 'no'}",
]
print("\n".join(info))
with open("/kaggle/working/uwct_logs/code_version.txt", "w") as f:
    f.write("\n".join(info) + "\n")

In [ ]:
# Cell 7 - raw data check (CLAUDE.md section 4): extracted files vs committed fingerprints
#   biology : structural fingerprint (D-011)  -> paths + sizes (+ txt/xlsx/docx hashes)
#   Harsh x3: full fingerprints (D-031)        -> sha256 of every file (--full)
# Every check must end with "RAW DATA MATCHES FINGERPRINT".

import subprocess, sys

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
FP = f"{REPO}/data/processed"
CHECKS = [
    ("biology",                 f"{FP}/raw_fingerprint_d011_post_cleanup.json",       []),
    ("harsh_model_a_2026-11",   f"{FP}/raw_fingerprint_harsh_model_a_2026-11.json",   ["--full"]),
    ("harsh_v1_extra_2026-11",  f"{FP}/raw_fingerprint_harsh_v1_extra_2026-11.json",  ["--full"]),
    ("harsh_v1_colour_2026-11", f"{FP}/raw_fingerprint_harsh_v1_colour_2026-11.json", ["--full"]),
]

report, all_ok = [], True
for folder, fingerprint, extra in CHECKS:
    r = subprocess.run([sys.executable, f"{REPO}/scripts/check_raw_unchanged.py",
                        "--root", f"{RAW}/{folder}", "--fingerprint", fingerprint, *extra],
                       capture_output=True, text=True)
    all_ok &= r.returncode == 0
    block = f"===== {folder}\n{r.stdout.strip()}" + (f"\n{r.stderr.strip()}" if r.stderr.strip() else "")
    print(block + "\n")
    report.append(block)

summary = "ALL 4 RAW CHECKS PASSED" if all_ok else "STOP: a raw check failed"
print(summary)
with open("/kaggle/working/uwct_logs/raw_check.txt", "w") as f:
    f.write("\n\n".join(report) + f"\n\n{summary}\n")


In [ ]:
# Cell 8 - install the pinned training package and record the environment (CLAUDE.md section 6)
# requirements-gpu.txt pins ultralytics==8.4.155 and says: do not reinstall torch.
# We print torch's version before and after to prove the install left it untouched.
# The environment report is saved to /kaggle/working/uwct_logs/env_report.json.

import subprocess, sys, json

REPO = "/tmp/uwct_repo"
torch_ver = lambda: subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                                   capture_output=True, text=True).stdout.strip()

before = torch_ver()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-gpu.txt"],
                   capture_output=True, text=True)
print("pip exit code:", r.returncode)
tail = (r.stdout + r.stderr).strip().splitlines()[-5:]
if tail:
    print("pip (last lines):\n  " + "\n  ".join(tail))
after = torch_ver()
print(f"torch before: {before}   after: {after}   {'UNCHANGED' if before == after else 'CHANGED - tell Claude'}")

r = subprocess.run([sys.executable, f"{REPO}/scripts/env_report.py",
                    "--out", "/kaggle/working/uwct_logs/env_report.json"],
                   capture_output=True, text=True)
print(r.stderr.strip() or "")
print(json.dumps(json.load(open("/kaggle/working/uwct_logs/env_report.json")), indent=1))

In [ ]:
# Cell 9 - build the YOLO view of frozen dataset v4 (D-032)
# Expected time: about 5-10 min.
# Raw roots point at the verified extracted data in /tmp/uwct_raw (read-only by the builder).
# Output goes to /tmp (not /kaggle/working: ~6 GB of image copies, rebuilt each session).
# The fingerprint must equal the one committed in the E007 config.

import json, os, subprocess, sys, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
OUT = "/tmp/uwct_yolo/lobster_mixed_v4"
EXPECTED_FP = "bbd4a8b172fd590b18d2e8c26be05c85cb3c3246a2e60a3b3513590b931cd13f"

env = dict(os.environ,
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")
t0 = time.time()
r = subprocess.run([sys.executable, f"{REPO}/scripts/prepare_yolo_dataset.py", "--version", "v4", "--out", OUT],
                   capture_output=True, text=True, env=env)
if r.returncode != 0:
    raise SystemExit("STOP: build failed:\n" + (r.stdout + r.stderr)[-3000:])

s = json.load(open(f"{OUT}/build_summary.json"))
print(f"built in {time.time() - t0:.0f} s   version {s['dataset_version']} ({s['version_status']})   "
      f"git {s['git']['commit'][:7]} dirty={s['git']['dirty']}")
for split, c in {**s["splits"], **s["eval_only_splits"]}.items():
    print(f"  {split:15s} images {c['images']:5d}   with lobster {c['positive']:5d}   "
          f"empty {c['negative']:4d}   boxes {c['instances']:6d}")
print("  not built:", sum(s["not_included"].values()), s["not_included"])
ok = s["dataset_fingerprint"] == EXPECTED_FP
print(f"\nfingerprint {s['dataset_fingerprint']}\n{'MATCHES the E007 config' if ok else 'STOP: differs from the E007 config'}")

os.system(f"cp {OUT}/build_summary.json /kaggle/working/uwct_logs/build_summary_v4.json")

In [ ]:
# Cell 10 - Ultralytics reads the built dataset (CLAUDE.md section 6: must print VERIFIED)
# Expected time: about 1-2 min (Ultralytics scans ~5,200 images and labels once).
# Checks per split that Ultralytics sees the same images / empty frames / boxes as build_summary.json,
# including the held-out Zenodo sk split (its own data file).

import subprocess, sys, time

REPO = "/tmp/uwct_repo"
OUT = "/tmp/uwct_yolo/lobster_mixed_v4"

t0 = time.time()
r = subprocess.run([sys.executable, f"{REPO}/scripts/verify_yolo_dataset.py", "--dataset", OUT],
                   capture_output=True, text=True)
print(r.stdout.strip())
print(f"\n(exit code {r.returncode}, {time.time() - t0:.0f} s)")
if r.returncode != 0:
    print("stderr (last lines):\n" + "\n".join(r.stderr.strip().splitlines()[-15:]))
with open("/kaggle/working/uwct_logs/verify_yolo_v4.txt", "w") as f:
    f.write(r.stdout)

In [ ]:
# Cell 11 - bring E027's finished run folder into /kaggle/working (read-only check first)
# Expected time: under 1 min.
# The round-2 notebook output (attached input) holds uwct_runs/E027 and E027_package.zip. We find the run folder
# (or unzip the package), check best.pt against the sha256 registered in experiments/registry.csv (D-043), and
# copy the folder to /kaggle/working/uwct_runs/E027, because the final test writes final_test*/ next to the run
# and /kaggle/input is read-only. last.pt is not needed and not copied.

import csv, glob, hashlib, os, shutil, zipfile

REPO = "/tmp/uwct_repo"
EXP = "E027"
RUNS = "/kaggle/working/uwct_runs"
DST = f"{RUNS}/{EXP}"

reg = {r["experiment_id"]: r for r in csv.DictReader(open(f"{REPO}/experiments/registry.csv"))}
EXPECTED_SHA = reg[EXP]["best_weights_sha256"]
print("registered best.pt sha256:", EXPECTED_SHA)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

if os.path.exists(DST):
    raise SystemExit(f"STOP: {DST} already exists - not overwriting.")

folders = [os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/uwct_runs/{EXP}/run_record.json", recursive=True)]
packages = glob.glob(f"/kaggle/input/**/{EXP}_package.zip", recursive=True)
print("run folders found:", folders)
print("packages found   :", packages)

src = None
for f in folders:
    best = f"{f}/train/weights/best.pt"
    if os.path.isfile(best) and sha256(best) == EXPECTED_SHA:
        src = f
        break
os.makedirs(RUNS, exist_ok=True)
if src:
    shutil.copytree(src, DST, ignore=shutil.ignore_patterns("last.pt"))
    print("copied from", src)
elif packages:
    tmp = "/tmp/e027_package"
    with zipfile.ZipFile(packages[0]) as z:
        names = [n for n in z.namelist() if n.startswith(f"uwct_runs/{EXP}/") and not n.endswith("last.pt")]
        assert all(".." not in n and not n.startswith("/") for n in names)
        z.extractall(tmp, members=names)
    shutil.copytree(f"{tmp}/uwct_runs/{EXP}", DST)
    print("unpacked from", packages[0])
else:
    raise SystemExit("STOP: E027's run folder/package not found - attach the round-2 notebook output as input.")

got = sha256(f"{DST}/train/weights/best.pt")
print("copied best.pt sha256   :", got)
if got != EXPECTED_SHA:
    raise SystemExit("STOP: best.pt differs from the registered sha256 - not the registered model.")
print("E027 RUN READY (best.pt MATCHES the registered sha256)")

In [ ]:
# Cell 12 - FINAL TEST EVALUATION of E027, target test (348 v1 test frames)
# Expected time: about 3-8 min (val pass for the operating confidence, then the 348 v1 test frames, then prediction examples).
# scripts/final_test_eval.py refuses (test split untouched) unless: GPU machine, clean checkout, E027 declared at this
# commit, valid full run record, best.pt sha256 unchanged, v4 view verifies with the training fingerprint, Ultralytics
# reads the target with the built counts, and no completed final test exists for this target.
# A refusal is printed and does NOT stop the notebook (the other target is independent).

import os, subprocess, sys, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
DATASET = "/tmp/uwct_yolo/lobster_mixed_v4"
RUNS = "/kaggle/working/uwct_runs"
LOG = "/kaggle/working/uwct_logs/final_test_E027_test.log"
RUN_FINAL_TEST = True   # this notebook exists only for the declared E027 final test (D-043)

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0",
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")
cmd = [sys.executable, f"{REPO}/scripts/final_test_eval.py", "--experiment", "E027", "--runs-root", RUNS,
       "--dataset", DATASET, "--target", "test", "--confirm-final-test"]

if RUN_FINAL_TEST:
    print("FINAL TEST EVALUATION -- E027 -- test")
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=REPO)
    with open(LOG, "w") as fh:
        fh.write(r.stdout + "\n--- stderr ---\n" + r.stderr)
    out = r.stdout.strip().splitlines()
    print("\n".join(out[-40:]))
    print(f"\n(exit code {r.returncode}, {time.time() - t0:.0f} s; full log {LOG})")
    if r.returncode != 0:
        print("!!! NOT EVALUATED (test): see the message above / stderr tail below")
        print("\n".join(r.stderr.strip().splitlines()[-15:]))

In [ ]:
# Cell 13 - FINAL TEST EVALUATION of E027, target test_zenodo_sk (428 held-out Zenodo sk frames)
# Expected time: about 3-8 min (val pass for the operating confidence, then the 428 held-out Zenodo sk frames, then prediction examples).
# scripts/final_test_eval.py refuses (test split untouched) unless: GPU machine, clean checkout, E027 declared at this
# commit, valid full run record, best.pt sha256 unchanged, v4 view verifies with the training fingerprint, Ultralytics
# reads the target with the built counts, and no completed final test exists for this target.
# A refusal is printed and does NOT stop the notebook (the other target is independent).

import os, subprocess, sys, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
DATASET = "/tmp/uwct_yolo/lobster_mixed_v4"
RUNS = "/kaggle/working/uwct_runs"
LOG = "/kaggle/working/uwct_logs/final_test_E027_test_zenodo_sk.log"
RUN_FINAL_TEST = True   # this notebook exists only for the declared E027 final test (D-043)

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0",
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")
cmd = [sys.executable, f"{REPO}/scripts/final_test_eval.py", "--experiment", "E027", "--runs-root", RUNS,
       "--dataset", DATASET, "--target", "test_zenodo_sk", "--confirm-final-test"]

if RUN_FINAL_TEST:
    print("FINAL TEST EVALUATION -- E027 -- test_zenodo_sk")
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=REPO)
    with open(LOG, "w") as fh:
        fh.write(r.stdout + "\n--- stderr ---\n" + r.stderr)
    out = r.stdout.strip().splitlines()
    print("\n".join(out[-40:]))
    print(f"\n(exit code {r.returncode}, {time.time() - t0:.0f} s; full log {LOG})")
    if r.returncode != 0:
        print("!!! NOT EVALUATED (test_zenodo_sk): see the message above / stderr tail below")
        print("\n".join(r.stderr.strip().splitlines()[-15:]))

In [ ]:
# Cell 14 - summary + one package of the final-test outputs (kept when the version is saved)
# Expected time: under 1 min. Claude re-reads final_test_record.json locally before anything is logged.

import hashlib, json, os, zipfile

W = "/kaggle/working"
RUN = f"{W}/uwct_runs/E027"
rows = []
for target, folder in (("test", "final_test"), ("test_zenodo_sk", "final_test_zenodo_sk")):
    p = f"{RUN}/{folder}/final_test_record.json"
    if not os.path.isfile(p):
        rows.append(f"{target:15s} | NOT EVALUATED (no record)")
        continue
    rec = json.load(open(p))
    u, a = rec["metrics"]["ultralytics_reported"], rec["metrics"]["at_val_conf"] or {}
    rows.append(f"{target:15s} | mAP50-95 {u['map50_95']:.4f} | mAP50 {u['map50']:.4f} | P {u['precision']:.4f} | "
                f"R {u['recall']:.4f} | at val conf {a.get('conf', float('nan')):.3f}: P {a.get('precision', float('nan')):.4f} "
                f"R {a.get('recall', float('nan')):.4f} F1 {a.get('f1', float('nan')):.4f} | commit {rec['evaluation_git']['commit'][:7]}")
print("E027 FINAL TEST (each target evaluated once)\n" + "\n".join(rows))

pkg = f"{W}/E027_final_test_package.zip"
with zipfile.ZipFile(pkg, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("final_test", "final_test_zenodo_sk"):
        for root, _, files in os.walk(f"{RUN}/{folder}"):
            for f in files:
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, W))
    for f in sorted(os.listdir(f"{W}/uwct_logs")):
        z.write(f"{W}/uwct_logs/{f}", f"uwct_logs/{f}")
h = hashlib.sha256(open(pkg, "rb").read()).hexdigest()
open(f"{W}/final_test_sha256.txt", "w").write(f"{h}  E027_final_test_package.zip\n")
print(f"\npackage {os.path.getsize(pkg):,} B sha256 {h}")